In [112]:
import pandas as pd


base_file_path = "../../data/raw/"
save_file_path = "../../data/iterim/"


def convert_to_integer(column: pd.Series):
    converted_column = column.astype("string").str.replace(" ", "").str.replace(",", ".").str.replace("\u00a0", "", regex=False).str.replace("—", "-", regex=False)
    converted_colum = pd.to_numeric(converted_column , errors="coerce")
    return converted_colum.round(2)


def convert_to_datetime(column: pd.Series):
    converted_column = pd.to_datetime(column, errors="coerce", format="mixed")
    return converted_column


def convert_to_string(column: pd.Series):
    converted_column = column.astype("string").str.strip().str.replace(" ", "").str.lower()
    return converted_column

In [113]:
cores_df = pd.read_csv(f"{base_file_path}cores.csv")

In [114]:
display(cores_df.describe())
display(cores_df.info())
display(cores_df)

,well_id,z_top,z_bottom,lithology,k_estimate_m_day,porosity
count,987,987,987,977,977,987
unique,403,5,6,6,939,6
top,BH-177,0,-20,Sand,22.47,0.26
freq,4,330,322,458,3,304


<class 'pandas.DataFrame'>
RangeIndex: 987 entries, 0 to 986
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   well_id           987 non-null    str  
 1   z_top             987 non-null    str  
 2   z_bottom          987 non-null    str  
 3   lithology         977 non-null    str  
 4   k_estimate_m_day  977 non-null    str  
 5   porosity          987 non-null    str  
dtypes: str(6)
memory usage: 73.2 KB


None

,well_id,z_top,z_bottom,lithology,k_estimate_m_day,porosity
0,BH-240,-46,-76,Silty sand,17.758,0.27
1,BH-295,-46,-76,Silty sand,17.666,0.27
2,BH-242,-20,-46,Sand,30.07,0.23
3,BH-093,0,-20,Gravel,59.466,0.26
4,BH-188,0,-20,Gravel,56.061,0.26
...,...,...,...,...,...,...
982,BH-159,-46,-76,Silty sand,17.597,0.27
983,BH-182,-20,−46,Sand,"29,313",0.23
984,BH-104,-20,-46,Gravel,59.527,"0,23"
985,BH-221,-20,-46,Sand,38.375,0.23


In [115]:
cores_df["lithology"] = convert_to_string(cores_df["lithology"])

cores_df["z_top"] = convert_to_integer(cores_df["z_top"])
cores_df["z_bottom"] = convert_to_integer(cores_df["z_bottom"])
cores_df["k_estimate_m_day"] = convert_to_integer(cores_df["k_estimate_m_day"])
cores_df["porosity"] = convert_to_integer(cores_df["porosity"])

In [116]:
cores_df[cores_df["well_id"].isin(["bh-146", "bh-144"])]

,well_id,z_top,z_bottom,lithology,k_estimate_m_day,porosity


In [117]:
display(cores_df.loc[cores_df["z_top"].isna(), ["well_id", "z_top"]])

,well_id,z_top
123,BH-146,<NA>
206,BH-144,<NA>
305,BH-219,<NA>
431,BH-306,<NA>
439,BH-287,<NA>
546,BH-229,<NA>
590,BH-247,<NA>
591,BH-062,<NA>
619,BH-134,<NA>
673,BH-134,<NA>


In [118]:
cores_df = cores_df.sort_values(["well_id", "z_top", "z_bottom"], ascending=[True, False, True])

In [119]:
cores_df

,well_id,z_top,z_bottom,lithology,k_estimate_m_day,porosity
840,BH-023,-20,-46,gravel,69.61,0.23
385,BH-036,-46,-76,siltysand,21.22,0.27
366,BH-039,-46,<NA>,sand,35.39,0.27
961,BH-047,0,-20,gravel,54.26,0.26
461,BH-054,-20,-46,sand,30.22,0.23
...,...,...,...,...,...,...
184,bh-279,-20,-46,siltysand,18.85,0.23
295,bh-281,-20,-46,sand,28.4,0.23
214,bh-281,-46,-76,siltysand,23.39,0.27
237,bh-288,-20,-46,siltysand,22.84,0.23


In [120]:
cores_df.groupby("well_id")

In [121]:
cores_df["z_bottom"] = cores_df["z_bottom"].fillna(cores_df.groupby("well_id")["z_top"].shift(-1))
cores_df["z_top"] = cores_df["z_top"].fillna(cores_df.groupby("well_id")["z_bottom"].shift(1))

In [122]:
cores_df.loc[cores_df["z_top"] == -46, "z_bottom"] = -76
cores_df.loc[cores_df["z_top"] == -20, "z_bottom"] = -46

In [123]:
cores_df.loc[cores_df["z_top"] == -76, "z_bottom"] = -76
cores_df.loc[cores_df["z_top"] == -76, "z_top"] = -46

In [129]:
cores_df["lithology"] = cores_df["lithology"].fillna("sand")

In [135]:
cores_df = cores_df[~cores_df["k_estimate_m_day"].isna()]

In [136]:
cores_df = cores_df.drop_duplicates(subset=["z_top", "z_bottom", "well_id"], keep="first")

In [138]:
cores_df.to_parquet(f"{save_file_path}cores.parquet")